# Mini-Project 5 — Audit an MCP Server
### IT7075: Applied AI for Cybersecurity · Model Context Protocol

MCP is a contract. A server advertises tools; a client — or an LLM agent — discovers them and calls
them, trusting whatever comes back. So the interesting question is not *can you build a server*, it
is **what happens when the tool is quietly wrong**.

You have a working MCP server with a recon tool, a CVE-lookup tool, and a knowledge-base resource.
It has real defects. Your job is to find them by measurement, fix what code can fix, and say what
code cannot.

**MCP has two sides, so your five TODOs are split across two files — 12 lines in total:**

| | Where | What | Lines |
|---|---|---|---|
| **TODO 1** | `project_mcp_server.py` | a version-aware `lookup_cve_v2` tool | 2 |
| **TODO 2** | `project_mcp_server.py` | expose the KB as a **resource** | 1 |
| **TODO 3** | this notebook | connect over stdio and **discover** the tools | 3 |
| **TODO 4** | this notebook | **call** a tool | 3 |
| **TODO 5** | this notebook | **read** the resource | 3 |

Find them by searching for `###################` in **both** files.

> **Runs fully offline.** Simulated scans, a local knowledge base, no LLM — so no API key, no
> model, no GPU, and no packets on the network. The numbers you report are exact.

> **The server restarts every time.** The client launches `project_mcp_server.py` as a fresh
> subprocess on each connection, so edit the server file, re-run a client cell, and your change is
> live. Nothing to restart.

## Part 1 — Setup: running async MCP code inside Jupyter

MCP's client API is asynchronous, and Jupyter already owns an event loop. `run_async` — the helper
from the Lecture 3 notebook — runs a coroutine on a worker thread with its own loop. Given to you;
nothing to change.

In [1]:
import sys, os, json, re, asyncio, threading
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


def run_async(coro):
    """GIVEN (Lecture 3): run a coroutine to completion on a worker thread,
    because Jupyter already owns the main event loop."""
    box = {}

    def worker():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            box["value"] = loop.run_until_complete(coro)
        except Exception as e:                       # noqa: BLE001
            box["error"] = e
        finally:
            loop.close()

    t = threading.Thread(target=worker)
    t.start()
    t.join()
    if "error" in box:
        raise box["error"]
    return box["value"]


# ---- where things live ---------------------------------------------------
def project_path(name):
    """The project files sit next to this notebook, or one level up from solutions/."""
    return name if os.path.exists(name) else os.path.join("..", name)


SERVER_FILE = project_path("project_mcp_server.py")
EXPECTED_FILE = project_path("expected_cves.json")

# How the client starts the server: as a subprocess, over stdio.
server = StdioServerParameters(command=sys.executable, args=[SERVER_FILE])
ERRLOG = open("mcp_server_stderr.log", "w")     # a real file: Jupyter dislikes pipes

print("server file :", SERVER_FILE)
print("python      :", sys.executable)

server file : project_mcp_server.py
python      : /Users/shamak/Desktop/UC Grad Sem 1/IT 7075- Applied AI for Cybersecurity/Mini Project MCP/.venv/bin/python


## Part 2 — Connect and discover  ·  **TODO 3**

This is the MCP handshake, and it is the whole point of the protocol: the client does not know what
the server can do until it asks. Read the descriptions it prints — that text is the entire contract
an LLM would use to decide which tool to call.

> **Expected output:** four tools — `port_scan`, `list_targets`, `lookup_cve`, `lookup_cve_v2`.

In [2]:
###################  TODO 3 — connect over stdio and list the tools  ###################
# WHAT: finish discover() so it returns [(tool name, first line of its docstring), ...]
#
# HOW (the Lecture 3 "Connect & Discover" cell) -- three lines inside the two
# `async with` blocks that are already open for you:
#
#   1. await session.initialize()                  the MCP handshake
#   2. tools = await session.list_tools()          ask what the server exposes
#   3. return [(t.name, t.description.splitlines()[0]) for t in tools.tools]
#
# Note `tools.tools`: list_tools() returns a result OBJECT with a .tools list on it.
################

async def discover():
    async with stdio_client(server, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            return [(t.name, t.description.splitlines()[0]) for t in tools.tools]


for name, description in run_async(discover()):
    print("- %-16s %s" % (name, description))

- port_scan        Run a (simulated) network port scan of a target host and return open ports
- list_targets     List the demo hosts this server can scan (simulated lab environment).
- lookup_cve       Look up known vulnerabilities for a service or product name (e.g.
- lookup_cve_v2    Look up known vulnerabilities for a service, matching on PRODUCT and
- assess_finding   Assess one service banner against the local CVE knowledge base and report


## Part 3 — Call a tool and read the resource  ·  **TODO 4 and TODO 5**

Two different MCP primitives. A **tool** is something the agent *does* (`call_tool`); a **resource**
is something it *reads* (`read_resource`). Same server, same session, different verbs.

> **Expected output:** a scan of `10.0.0.5` listing vsftpd, OpenSSH and Apache; then the first lines
> of the CVE knowledge base.

In [3]:
###################  TODO 4 — call a tool  ###################
# WHAT: run the `port_scan` tool on `target` and keep its text in `scan`.
#
# HOW (the Lecture 3 "Call the Tools" cell):
#       result = await session.call_tool("port_scan", {"target": target})
#       scan   = result.content[0].text
#
# The second argument is a DICT of the tool's parameters, keyed by name -- exactly
# the names in the server's function signature. Getting one wrong is the most common
# MCP client bug, and the server will tell you so.
################

###################  TODO 5 — read a resource  ###################
# WHAT: fetch the knowledge base the server exposes at the URI "kb://cve" into `kb`.
#
# HOW:  kb = await session.read_resource("kb://cve")
#       ...and the text is then kb.contents[0].text (see the return line below).
#
# Note the difference from a tool call: read_resource takes a URI and no arguments,
# and the payload is under .contents (plural) rather than .content.
################

async def scan_and_read(target):
    async with stdio_client(server, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("port_scan", {"target": target})
            scan = result.content[0].text
            kb = await session.read_resource("kb://cve")
            return scan, kb.contents[0].text


scan_text, kb_text = run_async(scan_and_read("10.0.0.5"))
print(scan_text)
print("\nRESOURCE kb://cve — first 180 characters:")
print(kb_text[:180], "...")

Nmap scan report for 10.0.0.5
PORT     STATE SERVICE  VERSION
21/tcp   open  ftp      vsftpd 2.3.4
22/tcp   open  ssh      OpenSSH 7.2p2 Ubuntu
80/tcp   open  http     Apache httpd 2.4.49

RESOURCE kb://cve — first 180 characters:
# vsftpd 2.3.4
The vsftpd 2.3.4 FTP server (commonly seen on port 21) shipped with a malicious backdoor introduced into the source distribution. Connecting with a username ending i ...


## Part 4 — Sweep the lab  *(no TODO — run it and read it)*

The coding is done; everything from here is measurement. Scan all four hosts, then ask **both**
lookup tools about every service that turned up.

> **Expected output:** 11 services across the four hosts.

In [4]:
SERVICE_LINE = re.compile(r"^\s*(\d+)/tcp\s+open\s+(\S+)\s+(.*)$")


def parse_services(scan_text):
    """GIVEN: pull (port, service, version-banner) out of nmap-style output."""
    found = []
    for line in scan_text.splitlines():
        m = SERVICE_LINE.match(line)
        if m:
            found.append((m.group(1), m.group(2), m.group(3).strip()))
    return found


async def sweep(hosts, tool_names):
    """GIVEN: one session; scan every host, then ask each lookup tool about
    every service banner found."""
    out = []
    async with stdio_client(server, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            for host in hosts:
                scan = (await session.call_tool("port_scan", {"target": host})).content[0].text
                for port, svc, banner in parse_services(scan):
                    row = {"host": host, "port": port, "banner": banner}
                    for tool in tool_names:
                        res = await session.call_tool(tool, {"service": banner})
                        row[tool] = res.content[0].text
                    out.append(row)
    return out


HOSTS = ["10.0.0.5", "10.0.0.8", "10.0.0.12", "10.0.0.20"]
rows = run_async(sweep(HOSTS, ["lookup_cve", "lookup_cve_v2"]))
print("discovered %d services across %d hosts\n" % (len(rows), len(HOSTS)))
for r in rows:
    print("  %-10s %-5s %s" % (r["host"], r["port"], r["banner"]))

discovered 11 services across 4 hosts

  10.0.0.5   21    vsftpd 2.3.4
  10.0.0.5   22    OpenSSH 7.2p2 Ubuntu
  10.0.0.5   80    Apache httpd 2.4.49
  10.0.0.8   139   Samba smbd 3.5.0
  10.0.0.8   3306  MySQL 5.5.40
  10.0.0.12  23    Linux telnetd
  10.0.0.12  443   Apache httpd 2.2.8 (OpenSSL 1.0.1)
  10.0.0.12  6379  Redis key-value store 4.0.9
  10.0.0.20  21    ProFTPD 1.3.5
  10.0.0.20  80    nginx 1.18.0
  10.0.0.20  5432  PostgreSQL 9.6.24


## Part 5 — Score both tools  *(no TODO)*

`expected_cves.json` records, for every service in the lab, which knowledge-base section *should*
come back — or `null` where the KB genuinely has no entry. Comparing each tool's answer with that
sorts every response into one of four buckets:

| | Meaning | Why it matters |
|---|---|---|
| **grounded** | the right advisory came back | what you want |
| **blind spot** | "no known CVEs" — but the KB *does* cover it | the agent is reassured, wrongly |
| **wrong version** | an advisory for a *different* version | worse than silence: it looks authoritative |
| **true gap** | "no known CVEs" and the KB really has nothing | honest — fix it with knowledge, not code |

> **Expected output:**
>
> | tool | grounded | blind spot | wrong version | true gap |
> |---|---:|---:|---:|---:|
> | `lookup_cve` (the lecture's) | 6/9 | **2** | **1** | 2 |
> | `lookup_cve_v2` (yours) | **9/9** | 0 | 0 | 2 |
>
> (11 services in all: 9 the knowledge base can cover, plus 2 it genuinely cannot.)
>
> Look at *which* ones the lecture version got wrong. `Linux telnetd` and
> `Redis key-value store 4.0.9` were **already in the knowledge base** — it simply failed to match
> them, and told the agent there was nothing to worry about. And `Apache httpd 2.2.8` came back with
> the advisory for **2.4.49**, a version that host is not running.
>
> Note *how* that last one fails. The naive tool returns the right OpenSSL advisory **and** the
> 2.4.49 one together — so it is not simply wrong, it is right with a fabricated extra finding
> attached. An analyst chasing a path-traversal bug on a server that cannot have it has lost just as
> much time as one who missed something.
>
> Your version fixes all three, and the two remaining gaps (nginx, PostgreSQL) are honest: the
> knowledge base has no entry. **No amount of matching logic fixes those** — that is Part 6.

In [5]:
GROUND_TRUTH = {s["service"]: s for s in json.load(open(EXPECTED_FILE))["services"]}


def classify(answer, service):
    """GIVEN: sort one tool answer into a bucket, given the ground truth.

    Note the `must_not` check: an answer can contain the right advisory AND an
    advisory for a version the host is not running. That still counts as a wrong
    version -- a report that names a vulnerability which is not there costs an
    analyst just as much time as one that misses a real one.
    """
    truth = GROUND_TRUTH.get(service, {})
    expected = truth.get("expected")
    said_nothing = answer.startswith("No known CVEs")
    if expected is None:
        return "true gap" if said_nothing else "wrong version"
    if said_nothing:
        return "blind spot"
    for forbidden in truth.get("must_not", []):
        if forbidden.lower() in answer.lower():
            return "wrong version"
    return "grounded" if expected.lower() in answer.lower() else "wrong version"


BUCKETS = ["grounded", "blind spot", "wrong version", "true gap"]
summary = {}
for tool in ["lookup_cve", "lookup_cve_v2"]:
    counts = dict.fromkeys(BUCKETS, 0)
    for r in rows:
        r[tool + "_verdict"] = classify(r[tool], r["banner"])
        counts[r[tool + "_verdict"]] += 1
    summary[tool] = counts

print("%-16s %-9s %-11s %-14s %s" % ("tool", "grounded", "blind spot", "wrong version", "true gap"))
for tool, c in summary.items():
    print("%-16s %-9d %-11d %-14d %d"
          % (tool, c["grounded"], c["blind spot"], c["wrong version"], c["true gap"]))

print("\nwhere they disagree:")
print("%-38s %-15s %s" % ("service", "lookup_cve", "lookup_cve_v2"))
for r in rows:
    if r["lookup_cve_verdict"] != r["lookup_cve_v2_verdict"]:
        print("%-38s %-15s %s" % (r["banner"][:36], r["lookup_cve_verdict"], r["lookup_cve_v2_verdict"]))

with open("results.csv", "w", encoding="utf-8") as f:
    f.write("host,port,service,lookup_cve,lookup_cve_v2\n")
    for r in rows:
        f.write("%s,%s,\"%s\",%s,%s\n" % (r["host"], r["port"], r["banner"],
                                          r["lookup_cve_verdict"], r["lookup_cve_v2_verdict"]))
print("\nwrote results.csv")

tool             grounded  blind spot  wrong version  true gap
lookup_cve       8         2           1              0
lookup_cve_v2    11        0           0              0

where they disagree:
service                                lookup_cve      lookup_cve_v2
Linux telnetd                          blind spot      grounded
Apache httpd 2.2.8 (OpenSSL 1.0.1)     wrong version   grounded
Redis key-value store 4.0.9            blind spot      grounded

wrote results.csv


> **Note (mine).** This notebook is saved *after* Part 6, so the knowledge base already holds my four new
> sections and the table above shows the after-knowledge numbers: the lecture tool 8 / 2 / 1 / 0 and
> `lookup_cve_v2` 11 / 0 / 0 / 0. Part 8 serves the course's original knowledge base to the same server
> and reproduces the expected table exactly (6 / 2 / 1 / 2 and 9 / 0 / 0 / 2), so the TODO 1 result is
> checked there, by an assertion.

## Part 6 — Close the real gaps  *(edit `project_cve_kb.md`, then re-run)*

Two services come back honest-but-empty: **nginx 1.18.0** and **PostgreSQL 9.6.24**. Better matching
cannot help — the knowledge base has nothing to match. This is the other half of a tool: its data.

**Add two sections to `project_cve_kb.md`**, in the same shape as the ones already there:

```markdown
# nginx 1.18.0
One or two sentences: what the software is, a real CVE or CWE that affects this
version, its severity, the fix, and the port it usually listens on.
```

Then re-run Parts 4 and 5. `true gap` should fall to 0 and `grounded` rise to 11/11.

> **Now be careful, because this lever cuts both ways.** Write a *vague* entry — a bare `# nginx`
> with no version — and it will match every nginx banner ever, including versions that were never
> vulnerable. Try it deliberately: make one entry too generic, re-run, and watch a `wrong version`
> appear. Report what you saw. **A tool is its logic, its data, and its docstring — and all three
> can lie.**

In [6]:
rows = run_async(sweep(HOSTS, ["lookup_cve", "lookup_cve_v2"]))

print("%-16s %-9s %-11s %-14s %s" % ("tool", "grounded", "blind spot", "wrong version", "true gap"))
for tool in ["lookup_cve", "lookup_cve_v2"]:
    counts = dict.fromkeys(BUCKETS, 0)
    for r in rows:
        counts[classify(r[tool], r["banner"])] += 1
    print("%-16s %-9d %-11d %-14d %d"
          % (tool, counts["grounded"], counts["blind spot"],
             counts["wrong version"], counts["true gap"]))

print("\n(If nothing changed, you have not saved project_cve_kb.md yet — the server")
print(" re-reads it on every call, so no restart is needed.)")

tool             grounded  blind spot  wrong version  true gap
lookup_cve       8         2           1              0
lookup_cve_v2    11        0           0              0

(If nothing changed, you have not saved project_cve_kb.md yet — the server
 re-reads it on every call, so no restart is needed.)


> **What I added (mine).** Four sections at the end of `project_cve_kb.md`, all cited in `SOURCES.md`:
> `nginx 1.18.0` (CVE-2021-23017), `PostgreSQL 9.6` (end of life since November 2021; no single CVE, the
> same way the course's Redis entry has none), and two for my own host, `OpenSSH 8.9p1` (CVE-2024-6387)
> and `Apache httpd 2.4.50` (CVE-2021-42013). `expected_cves.json` now expects the nginx and PostgreSQL
> sections, which is why the table above shows 0 true gaps.

## Part 7 — Audit what the server exposes  *(no TODO)*

One last MCP-specific question, straight from the Lecture 3 safety section. An agent will call
anything a server advertises, so **the advertised surface is the attack surface**. Look at what this
server hands to any client that connects.

In [7]:
async def audit():
    async with stdio_client(server, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = (await session.list_tools()).tools
            resources = (await session.list_resources()).resources
            return tools, resources


tools, resources = run_async(audit())
print("This server advertises %d tools and %d resource(s):\n" % (len(tools), len(resources)))
for t in tools:
    params = list((t.inputSchema or {}).get("properties", {}))
    doc = (t.description or "").strip().splitlines()
    print("  %-16s params=%-22s doc lines=%d" % (t.name, ",".join(params) or "none", len(doc)))
    print("      %s" % (doc[0] if doc else "(NO DESCRIPTION -- an agent is guessing)"))
for r in resources:
    print("  resource %s  (%s)" % (r.uri, r.name))

print("\nFor your write-up, answer these from the output above:")
for q in ["which tools take free-text input from the model, and what would a bad value do?",
          "does any tool touch the network or the filesystem, and how would a client know?",
          "is any description vague enough that an agent could call the wrong tool?",
          "what would you refuse to expose over MCP at all?"]:
    print("  -", q)

This server advertises 5 tools and 1 resource(s):

  port_scan        params=target                 doc lines=4
      Run a (simulated) network port scan of a target host and return open ports
  list_targets     params=none                   doc lines=1
      List the demo hosts this server can scan (simulated lab environment).
  lookup_cve       params=service                doc lines=7
      Look up known vulnerabilities for a service or product name (e.g.
  lookup_cve_v2    params=service                doc lines=5
      Look up known vulnerabilities for a service, matching on PRODUCT and
  assess_finding   params=service                doc lines=23
      Assess one service banner against the local CVE knowledge base and report
  resource kb://cve  (cve_kb)

For your write-up, answer these from the output above:
  - which tools take free-text input from the model, and what would a bad value do?
  - does any tool touch the network or the filesystem, and how would a client know?
  - i

**My answers to the audit questions (mine).**

- *Free-text input.* `port_scan(target)` and the three lookup tools (`lookup_cve`, `lookup_cve_v2`, `assess_finding`, each taking `service`) accept any string a model writes. Here the damage is limited, because the scan is canned and the lookups only read the knowledge base. Behind the commented-out real `nmap` call, though, `target` decides which host gets scanned, so a model that was told to "scan everything", or that was fed a crafted address, would scan a host nobody authorised. A real version needs an allow-list of targets.
- *Network and filesystem.* None of these tools touch the network. The lookups read `project_cve_kb.md` from disk on every call. A client cannot tell this from the protocol, only from what each docstring says, and only `port_scan` (SIMULATED) and `assess_finding` (read only) say anything about it.
- *Vague descriptions.* `lookup_cve` and `lookup_cve_v2` open with almost the same sentence, so a model has nothing to choose between them, and `lookup_cve` says nothing about the false matches it is known to make. `list_targets` is fine.
- *What I would not expose.* The lecture `lookup_cve`: it is kept only as a baseline, and it is wrong in both directions (blind spots and wrong versions). And a real `port_scan` with no target allow-list.

## Part 8: your own server, your own tool, your own measurement

Everything above used the provided server and the provided lab, and it was practice. The rest of this
notebook is your own, and no code is provided for it.

First extend the server. In `project_mcp_server.py`:

1. Add one host of your own to `_SIMULATED_HOSTS`, with three realistic service banners, product and
   version in nmap style. Choose them so at least one is covered by your knowledge base and at least
   one is not.
2. Add a tool of your own that an analyst would find useful, for example one that reports the
   severity of a finding, checks a port against a policy list, or summarizes what a host exposes.
   Write its docstring as carefully as its code: the docstring is what a client and a model read.
3. Add the missing knowledge to `project_cve_kb.md` for the services that return nothing, with real
   products and real CVEs, cited in `SOURCES.md`, and record the expected section for each of your
   services in `expected_cves.json`.

Then write the client cells below to:

4. Connect to your server, discover the tools, and show that your new tool is advertised.
5. Sweep your own host with both lookup tools and with your new tool, and sort every answer into the
   four buckets: grounded, blind spot, wrong version, and true gap. Save the table as
   `results_mine.csv`.
6. Make one knowledge base entry deliberately vague by dropping the version, run the sweep again, and
   record what changed.

> When you are finished, run the whole notebook top to bottom and save it with the outputs showing,
> then commit it, the server, the knowledge base, and your results to your private GitHub repository.
> See `MiniProject_MCP.md` section 6.1 for the folder structure.

### 8.1 Connect and discover (my client)

I added host `10.0.0.33` and a tool, `assess_finding`, to `project_mcp_server.py`. The client starts the
server as a fresh subprocess and asks it what it offers. The description printed below is the whole of
what a model would be told about my tool.

In [8]:
###################  YOUR SERVER: CONNECT AND DISCOVER  ###################
# My client, written from scratch: connect over stdio, run the handshake, and see
# what the extended server advertises. Nothing here knows the tool names in advance.
async def discover_full(params):
    async with stdio_client(params, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            init = await session.initialize()
            tools = (await session.list_tools()).tools
            resources = (await session.list_resources()).resources
            return init.serverInfo, tools, resources


info, my_tools, my_resources = run_async(discover_full(server))
print("connected to %s (protocol server version %s)" % (info.name, info.version))
print("advertised tools:", ", ".join(t.name for t in my_tools))
print("advertised resources:", ", ".join(str(r.uri) for r in my_resources))

mine = next(t for t in my_tools if t.name == "assess_finding")
print("\n--- what a client (or a model) is told about assess_finding ---")
print("input schema:", json.dumps(mine.inputSchema))
print("description:\n")
print(mine.description)   # exactly the text the server sends, indentation and all


connected to cyber-recon-project (protocol server version 1.30.0)
advertised tools: port_scan, list_targets, lookup_cve, lookup_cve_v2, assess_finding
advertised resources: kb://cve

--- what a client (or a model) is told about assess_finding ---
input schema: {"properties": {"service": {"title": "Service", "type": "string"}}, "required": ["service"], "title": "assess_findingArguments", "type": "object"}
description:

Assess one service banner against the local CVE knowledge base and report
    each matching advisory with its CVE, severity and fix, most severe first.

    Call it once per open port after port_scan, passing the product-and-version
    part of the scan line exactly as printed, for example
    'Apache httpd 2.4.50 ((Unix))' or 'OpenSSH 8.9p1 Ubuntu 3ubuntu0.6'.
    Read only: it reads the local knowledge base and nothing else. It sends no
    packets and writes no files.

    How to read the answer:
      - 'match: version-confirmed' means the entry's product name and its

### 8.2 Call my tool on my host

`port_scan` first, then one `assess_finding` call for each service it reports, all in one session.

In [9]:
# Call my tool on my own host, through the protocol, the way an agent would:
# scan first, then one assess_finding call per service the scan reports.
async def scan_and_assess(params, target):
    async with stdio_client(params, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            scan = (await session.call_tool("port_scan", {"target": target})).content[0].text
            out = [scan]
            for port, svc, banner in parse_services(scan):
                res = await session.call_tool("assess_finding", {"service": banner})
                out.append(res.content[0].text)
            return out


MY_HOST = "10.0.0.33"
for block in run_async(scan_and_assess(server, MY_HOST)):
    print(block, end="\n\n")


Nmap scan report for 10.0.0.33
PORT     STATE SERVICE  VERSION
22/tcp   open  ssh      OpenSSH 8.9p1 Ubuntu 3ubuntu0.6
80/tcp   open  http     Apache httpd 2.4.50 ((Unix))
6379/tcp open  redis    Redis key-value store 4.0.14

Assessment of 'OpenSSH 8.9p1 Ubuntu 3ubuntu0.6': 1 advisory(ies), highest severity HIGH
- section: OpenSSH 8.9p1
  cve: CVE-2024-6387
  severity: high
  match: version-confirmed ('openssh' and '8.9p1' both in the banner)
  fix: upgrade to OpenSSH 9.8p1, or to a distribution build that carries the fix (Ubuntu 22.04: 1:8.9p1-3ubuntu0.10 or later, USN-6859-1); as a stopgap set LoginGraceTime 0 in sshd_config, which trades the code-execution risk for a denial-of-service risk.

Assessment of 'Apache httpd 2.4.50 ((Unix))': 1 advisory(ies), highest severity CRITICAL
- section: Apache httpd 2.4.50
  cve: CVE-2021-42013
  severity: critical
  match: version-confirmed ('apache' and '2.4.50' both in the banner)
  fix: upgrade to Apache 2.4.51 or later, keep "Require all den

### Your measurement

**Three knowledge bases, one server.** To give the before and after numbers without hand-editing a file
between runs, the server reads `CVE_KB_FILE` from its environment, a two-line change to `_kb_path()`.
Each sweep launches a fresh server on a different copy:

| state | what it is |
|---|---|
| `course` | the knowledge base as the course shipped it (my four sections removed) |
| `mine` | `project_cve_kb.md` as committed |
| `vague` | `mine`, with the version dropped from one title: `# OpenSSH 8.9p1` becomes `# OpenSSH` |

In [10]:
###################  YOUR MEASUREMENT  ###################
# Three versions of the knowledge base, served by the SAME server code.
# The server reads CVE_KB_FILE from its environment (a two-line change to _kb_path),
# so each sweep below launches a fresh server process on a different knowledge base:
#   course : the knowledge base as the course shipped it (my four sections removed)
#   mine   : project_cve_kb.md as committed, with my four sections
#   vague  : mine, but with the version dropped from one title: "# OpenSSH 8.9p1" -> "# OpenSSH"
import tempfile

KB_FILE = os.path.abspath(project_path("project_cve_kb.md"))
MY_SECTIONS = ["nginx 1.18.0", "PostgreSQL 9.6", "OpenSSH 8.9p1", "Apache httpd 2.4.50"]
VAGUE = {"OpenSSH 8.9p1": "OpenSSH"}          # old title -> vague title


def kb_sections_of(text):
    """The same split the server uses: {title: section}."""
    out = {}
    for chunk in text.split("#"):
        chunk = chunk.strip()
        if chunk:
            out[chunk.splitlines()[0].strip()] = chunk
    return out


full_text = open(KB_FILE, encoding="utf8").read()
sections = kb_sections_of(full_text)
assert all(t in sections for t in MY_SECTIONS), "my sections are missing from the KB"

tmp = tempfile.mkdtemp(prefix="kb_states_")
KB_STATES = {}
for state, titles in [
    ("course", [t for t in sections if t not in MY_SECTIONS]),
    ("mine", list(sections)),
    ("vague", list(sections)),
]:
    parts = []
    for t in titles:
        body = sections[t]
        if state == "vague" and t in VAGUE:
            body = VAGUE[t] + body[len(t):]
        parts.append("# " + body)
    path = os.path.join(tmp, f"kb_{state}.md")
    open(path, "w", encoding="utf8").write("\n\n".join(parts) + "\n")
    KB_STATES[state] = {
        "path": path,
        "titles": set(kb_sections_of(open(path, encoding="utf8").read())),
        "rename": VAGUE if state == "vague" else {},
        "server": StdioServerParameters(command=sys.executable, args=[SERVER_FILE],
                                        env={"CVE_KB_FILE": path}),
    }
    print("%-6s %2d sections  %s" % (state, len(KB_STATES[state]["titles"]), path))


course 12 sections  /var/folders/0j/tmfp03js3l96jnz45wmv8w1h0000gn/T/kb_states_5ck12lfy/kb_course.md
mine   16 sections  /var/folders/0j/tmfp03js3l96jnz45wmv8w1h0000gn/T/kb_states_5ck12lfy/kb_mine.md
vague  16 sections  /var/folders/0j/tmfp03js3l96jnz45wmv8w1h0000gn/T/kb_states_5ck12lfy/kb_vague.md


**Scoring.** Part 5's `classify` checks only the `must_not` list, so an answer that carries the right
advisory plus an unrelated one still counts as grounded. My scorer reads which sections came back, by
title, and calls an answer *grounded* only if the expected section is there and nothing else is (a
section in `also_ok` is allowed). Anything else counts as *wrong version*, which is the assignment's
definition: an advisory for a version this host is not running. First I check it against Part 5's scorer
on the course knowledge base.

In [11]:
# A stricter scorer than Part 5's. It reads WHICH sections came back, by title,
# instead of searching the answer text for substrings, and it applies the
# definition literally: any advisory for a version this host is not running makes
# the answer a wrong version, not only the ones listed under must_not.
# It is also knowledge-base aware: if the expected section is not in the
# knowledge base being served, a "no known CVEs" is a true gap, not a blind spot.
TRUTH = {s["service"]: s for s in json.load(open(EXPECTED_FILE))["services"]}


def returned_titles(answer):
    if answer.startswith("No known CVEs"):
        return set()
    mine = re.findall(r"^- section: (.+)$", answer, re.M)       # assess_finding format
    if mine:
        return set(mine)
    return {block.splitlines()[0].strip() for block in answer.split("\n\n")}


def classify_strict(answer, service, state):
    truth = TRUTH[service]
    rename, titles = KB_STATES[state]["rename"], KB_STATES[state]["titles"]
    expected = rename.get(truth["expected"], truth["expected"])
    if expected not in titles:
        expected = None
    ok = {expected} | {rename.get(t, t) for t in truth.get("also_ok", [])}
    got = returned_titles(answer)
    if not got:
        return "blind spot" if expected else "true gap"
    if expected is None:
        return "wrong version"
    return "grounded" if expected in got and got <= ok else "wrong version"


# Check the scorer against Part 5 before trusting it: on the course knowledge base
# it must give the starter's expected table exactly.
TOOLS = ["lookup_cve", "lookup_cve_v2", "assess_finding"]
ALL_HOSTS = HOSTS + [MY_HOST]


async def sweep_with(params, hosts, tool_names):
    """Like Part 4's sweep, but the server (and so the knowledge base) is a parameter."""
    out = []
    async with stdio_client(params, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            for host in hosts:
                scan = (await session.call_tool("port_scan", {"target": host})).content[0].text
                for port, svc, banner in parse_services(scan):
                    row = {"host": host, "port": port, "banner": banner}
                    for tool in tool_names:
                        row[tool] = (await session.call_tool(tool, {"service": banner})).content[0].text
                    out.append(row)
    return out


SWEEPS = {state: run_async(sweep_with(KB_STATES[state]["server"], ALL_HOSTS, TOOLS))
          for state in KB_STATES}


def table(state, hosts, tools=TOOLS):
    rows = [r for r in SWEEPS[state] if r["host"] in hosts]
    result = {}
    for tool in tools:
        c = dict.fromkeys(BUCKETS, 0)
        for r in rows:
            c[classify_strict(r[tool], r["banner"], state)] += 1
        result[tool] = c
    return result, len(rows)


def show(title, state, hosts, tools=TOOLS):
    res, n = table(state, hosts, tools)
    print("%s  (%d services)" % (title, n))
    print("  %-16s %9s %11s %14s %9s" % ("tool", "grounded", "blind spot", "wrong version", "true gap"))
    for tool, c in res.items():
        print("  %-16s %9d %11d %14d %9d" % (tool, c["grounded"], c["blind spot"],
                                            c["wrong version"], c["true gap"]))
    print()
    return res


# Check the strict scorer against Part 5's own scorer on the course knowledge base.
def classify_part5(answer, service, state):
    """Part 5's given classify(), made knowledge-base aware the same way."""
    t = TRUTH[service]
    saved = GROUND_TRUTH.get(service)
    GROUND_TRUTH[service] = {**t, "expected": t["expected"] if t["expected"] in KB_STATES[state]["titles"] else None}
    try:
        return classify(answer, service)
    finally:
        GROUND_TRUTH[service] = saved


course_rows = [r for r in SWEEPS["course"] if r["host"] in HOSTS]
for tool in ["lookup_cve", "lookup_cve_v2"]:
    c = dict.fromkeys(BUCKETS, 0)
    for r in course_rows:
        c[classify_part5(r[tool], r["banner"], "course")] += 1
    print("Part 5 scorer, course KB, %-14s %s" % (tool, [c[b] for b in BUCKETS]))
    assert [c[b] for b in BUCKETS] == {"lookup_cve": [6, 2, 1, 2], "lookup_cve_v2": [9, 0, 0, 2]}[tool]
print("(matches the starter's expected output)\n")

show("strict scorer, course lab, course knowledge base", "course", HOSTS, ["lookup_cve", "lookup_cve_v2"])

print("where the two scorers disagree:")
for r in course_rows:
    for tool in ["lookup_cve", "lookup_cve_v2"]:
        a, b = classify_part5(r[tool], r["banner"], "course"), classify_strict(r[tool], r["banner"], "course")
        if a != b:
            print("  %-22s %-14s Part 5: %-9s strict: %-14s returned %s"
                  % (r["banner"], tool, a, b, sorted(returned_titles(r[tool]))))


Part 5 scorer, course KB, lookup_cve     [6, 2, 1, 2]
Part 5 scorer, course KB, lookup_cve_v2  [9, 0, 0, 2]
(matches the starter's expected output)

strict scorer, course lab, course knowledge base  (11 services)
  tool              grounded  blind spot  wrong version  true gap
  lookup_cve               6           2              1         2
  lookup_cve_v2            8           0              1         2

where the two scorers disagree:
  Apache httpd 2.4.49    lookup_cve_v2  Part 5: grounded  strict: wrong version  returned ['Apache Log4j 2.14', 'Apache httpd 2.4.49']


The scorers disagree on one answer, and it is a real defect in TODO 1's rule. `lookup_cve_v2` reads the
title's version with `\d[\d.p]*`, which in `Apache Log4j 2.14` finds the **4 in "Log4j"**. And "4" appears
in `2.4.49`. So the Log4Shell advisory comes back for an Apache web server. Part 5's scorer cannot see
this because `Log4j` is not on that service's `must_not` list. `assess_finding` needs a whole version
number, and `.4.` is not one.

### 8.3 My host, before and after my knowledge

In [12]:
# My host, three tools, before and after my knowledge.
for state, label in [("course", "BEFORE: course knowledge base"),
                     ("mine", "AFTER: with my four sections")]:
    show("my host %s, %s" % (MY_HOST, label), state, [MY_HOST])

print("per service on my host:")
print("  %-34s %-7s %-14s %-14s %s" % ("service", "kb", *TOOLS))
for state in ["course", "mine"]:
    for r in SWEEPS[state]:
        if r["host"] == MY_HOST:
            print("  %-34s %-7s %-14s %-14s %s" % (
                r["banner"], state, *[classify_strict(r[t], r["banner"], state) for t in TOOLS]))


my host 10.0.0.33, BEFORE: course knowledge base  (3 services)
  tool              grounded  blind spot  wrong version  true gap
  lookup_cve               0           1              1         1
  lookup_cve_v2            1           0              1         1
  assess_finding           1           0              0         2

my host 10.0.0.33, AFTER: with my four sections  (3 services)
  tool              grounded  blind spot  wrong version  true gap
  lookup_cve               1           1              1         0
  lookup_cve_v2            2           0              1         0
  assess_finding           3           0              0         0

per service on my host:
  service                            kb      lookup_cve     lookup_cve_v2  assess_finding
  OpenSSH 8.9p1 Ubuntu 3ubuntu0.6    course  true gap       true gap       true gap
  Apache httpd 2.4.50 ((Unix))       course  wrong version  wrong version  true gap
  Redis key-value store 4.0.14       course  blind spot     gro

### 8.4 The whole lab, three knowledge bases

In [13]:
# The same thing over the whole lab, all 14 services.
for state, label in [("course", "course knowledge base"),
                     ("mine", "my knowledge base"),
                     ("vague", "my knowledge base with '# OpenSSH 8.9p1' made vague")]:
    show("whole lab, %s" % label, state, ALL_HOSTS)


whole lab, course knowledge base  (14 services)
  tool              grounded  blind spot  wrong version  true gap
  lookup_cve               6           3              2         3
  lookup_cve_v2            9           0              2         3
  assess_finding          10           0              0         4

whole lab, my knowledge base  (14 services)
  tool              grounded  blind spot  wrong version  true gap
  lookup_cve               8           3              3         0
  lookup_cve_v2           12           0              2         0
  assess_finding          14           0              0         0

whole lab, my knowledge base with '# OpenSSH 8.9p1' made vague  (14 services)
  tool              grounded  blind spot  wrong version  true gap
  lookup_cve               8           3              3         0
  lookup_cve_v2           11           0              3         0
  assess_finding          13           0              1         0



### 8.5 The vague entry, service by service

In [14]:
# What the vague entry changed, service by service.
print("services whose verdict changed when '# OpenSSH 8.9p1' became '# OpenSSH':\n")
print("  %-34s %-15s %-15s %s" % ("service", "tool", "mine", "vague"))
for before, after in zip(SWEEPS["mine"], SWEEPS["vague"]):
    for tool in TOOLS:
        b = classify_strict(before[tool], before["banner"], "mine")
        a = classify_strict(after[tool], after["banner"], "vague")
        if a != b:
            print("  %-34s %-15s %-15s %s" % (before["banner"], tool, b, a))

print("\nwhat assess_finding now says about the course host's OpenSSH 7.2p2:\n")
row = next(r for r in SWEEPS["vague"] if r["banner"].startswith("OpenSSH 7.2p2"))
print(row["assess_finding"])


services whose verdict changed when '# OpenSSH 8.9p1' became '# OpenSSH':

  service                            tool            mine            vague
  OpenSSH 7.2p2 Ubuntu               lookup_cve_v2   grounded        wrong version
  OpenSSH 7.2p2 Ubuntu               assess_finding  grounded        wrong version

what assess_finding now says about the course host's OpenSSH 7.2p2:

Assessment of 'OpenSSH 7.2p2 Ubuntu': 2 advisory(ies), highest severity HIGH
- section: OpenSSH
  cve: CVE-2024-6387
  severity: high
  match: UNCONFIRMED (the entry names no version, so it matches every 'openssh' banner; check the version by hand)
  fix: upgrade to OpenSSH 9.8p1, or to a distribution build that carries the fix (Ubuntu 22.04: 1:8.9p1-3ubuntu0.10 or later, USN-6859-1); as a stopgap set LoginGraceTime 0 in sshd_config, which trades the code-execution risk for a denial-of-service risk.
- section: OpenSSH 7.2p2
  cve: CVE-2018-15473
  severity: medium
  match: version-confirmed ('openssh' and '

### 8.6 Save the table

In [15]:
# results_mine.csv: one row per service, per knowledge-base state, with each tool's bucket.
import csv

with open("results_mine.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["kb_state", "host", "port", "service", "expected_section", *TOOLS])
    for state in KB_STATES:
        for r in SWEEPS[state]:
            truth = TRUTH[r["banner"]]
            exp = KB_STATES[state]["rename"].get(truth["expected"], truth["expected"])
            exp = exp if exp in KB_STATES[state]["titles"] else ""
            w.writerow([state, r["host"], r["port"], r["banner"], exp,
                        *[classify_strict(r[t], r["banner"], state) for t in TOOLS]])
print("wrote results_mine.csv: %d rows" % sum(len(v) for v in SWEEPS.values()))
print(open("results_mine.csv").read())


wrote results_mine.csv: 42 rows
kb_state,host,port,service,expected_section,lookup_cve,lookup_cve_v2,assess_finding
course,10.0.0.5,21,vsftpd 2.3.4,vsftpd 2.3.4,grounded,grounded,grounded
course,10.0.0.5,22,OpenSSH 7.2p2 Ubuntu,OpenSSH 7.2p2,grounded,grounded,grounded
course,10.0.0.5,80,Apache httpd 2.4.49,Apache httpd 2.4.49,grounded,wrong version,grounded
course,10.0.0.8,139,Samba smbd 3.5.0,Samba 3.5.0,grounded,grounded,grounded
course,10.0.0.8,3306,MySQL 5.5.40,MySQL 5.5,grounded,grounded,grounded
course,10.0.0.12,23,Linux telnetd,Telnet service,blind spot,grounded,grounded
course,10.0.0.12,443,Apache httpd 2.2.8 (OpenSSL 1.0.1),OpenSSL 1.0.1,wrong version,grounded,grounded
course,10.0.0.12,6379,Redis key-value store 4.0.9,Redis 4.0,blind spot,grounded,grounded
course,10.0.0.20,21,ProFTPD 1.3.5,ProFTPD 1.3.5,grounded,grounded,grounded
course,10.0.0.20,80,nginx 1.18.0,,true gap,true gap,true gap
course,10.0.0.20,5432,PostgreSQL 9.6.24,,true gap,true gap,true gap
course,10.0.0.33,22,

### 8.7 One probe outside the lab

TODO 1's version test is a plain substring check, so it fails at the edges as well as on Log4j.

In [16]:
# One more look at the matching lever. TODO 1's rule tests the version as a plain
# substring, so "4.0" is found inside "7.4.0". My tool requires a whole version.
# Redis 7.4.0 is a real release; it is not in the lab, so this is a probe, not a score.
async def ask(params, service):
    async with stdio_client(params, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return {t: (await session.call_tool(t, {"service": service})).content[0].text
                    for t in ["lookup_cve_v2", "assess_finding"]}


for probe in ["Redis key-value store 7.4.0"]:
    ans = run_async(ask(server, probe))
    for tool, text in ans.items():
        print("%-15s %-30s -> %s" % (tool, probe, sorted(returned_titles(text)) or "nothing"))


lookup_cve_v2   Redis key-value store 7.4.0    -> ['Redis 4.0']
assess_finding  Redis key-value store 7.4.0    -> nothing


### What I found

| whole lab, 14 services | `lookup_cve` | `lookup_cve_v2` | `assess_finding` |
|---|---|---|---|
| course knowledge base | 6 / 3 / 2 / 3 | 9 / 0 / 2 / 3 | 10 / 0 / 0 / 4 |
| + my four sections | 8 / 3 / 3 / 0 | 12 / 0 / 2 / 0 | **14 / 0 / 0 / 0** |
| + `# OpenSSH` made vague | 8 / 3 / 3 / 0 | 11 / 0 / 3 / 0 | 13 / 0 / 1 / 0 |

(grounded / blind spot / wrong version / true gap)

- **Matching fixed what it could and no more.** It turned both blind spots into grounded answers
  (telnetd and Redis 4.0.9), and on my host Redis 4.0.14 too. It could not answer what the knowledge
  base did not contain: nginx, PostgreSQL and my host's OpenSSH stayed true gaps under every matcher.
- **Knowledge fixed the gaps, and on a bad matcher it also did harm.** My four sections cleared every true
  gap. But `lookup_cve` started answering the course host's `OpenSSH 7.2p2 Ubuntu` with my regreSSHion
  entry as well, only because that entry mentions Ubuntu. Its wrong-version count went *up* from 2 to 3
  when I added correct knowledge.
- **The vague entry** cost `lookup_cve_v2` and `assess_finding` one grounded answer each: OpenSSH 7.2p2
  was now also told about CVE-2024-6387, which needs 8.5p1 or later. `assess_finding` was still wrong, but
  it labelled the match `UNCONFIRMED`. That label is the only signal a client gets that it should check.
- **TODO 1's rule has a false positive the course scorer misses** (Log4Shell for Apache httpd), which
  shows up on both Apache hosts.

In [17]:
###################  YOUR MEASUREMENT  ###################
# Sweep your own host, sort every answer into grounded, blind spot, wrong
# version, or true gap, and save the table as results_mine.csv. Then make one
# knowledge base entry vague, run it again, and record what changed.
################

---

## What to write up

One PDF report and a video of five to ten minutes. See `MiniProject_MCP.md` for the full
requirements. The questions that matter:

- The blind spot is the dangerous bucket. A tool that says no known CVEs about a service the
  knowledge base covers is worse than an error message. Why, and what would a client have to do to
  notice?
- A wrong version answer looks authoritative. Which of the two failures would you rather ship, and
  what does your answer say about how much an agent should trust a tool result?
- Two levers. What did better matching fix, what needed new knowledge, and what broke when you made
  an entry vague on purpose? Give the numbers before and after.
- The docstring is the contract. Which description on your server would you rewrite before letting a
  model use it, and which tool would you refuse to expose at all?
- A tool over MCP compared with a tool inside your agent. In the LangGraph module a tool was a Python
  function you called yourself. Here it is a separate process a client discovers. What did you gain,
  what did it cost, and when would you keep the function?

Submit one PDF report with the link to your project folder and the link to your video on its first
page. The notebook, the server, the knowledge base, and your results live in the repository, which is
private, with your instructor and the TA added as collaborators. The notebook must be committed with
its outputs showing.